<h1> Classificação Multilabel

A classificação multilabel diz respeito a uma classificação que pode ter mais de um resultado

As tres formas basicas de algoritmos para resolver esse problema são:
* Binare relevance
* Classifier Chains
* Label Powerset

Tendo como um exemplo um dataset em que os atribiutos sejam:
 X1 | X2 | X3 | Agitada(truo ou false) | alegre(true ou false) | Relaxante(true ou false)| Encantadora(true ou false)

é possivel que alfo seja alegre e encantador ao mesmo tempo, por exemplo.

Então a logica deles é a seguinte:
>O Binare relevance vai ir label a label classificar coluna a coluna para ver se está certo

>O Classifier Chains faz basicamente o mesmo, mas utiliza os dados já classificados anteriormente para a classificação posterior

>O label Powerset usa cada linha diferente de atributos como uma classe por si só

Quanto as metricas tambem existem as basicas: 
* Exact math: Considera apenas se todos os labels estiverem corretas 
* Hamming loss: Faz uma comparação entre certo e errado e dá o haming distance vendo qual a fração de erros para cada label


In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC # Algoritmo de classificação
from sklearn.metrics import hamming_loss 
from sklearn.multioutput import ClassifierChain
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier

In [2]:

data = pd.read_csv("C:/projetos/GitHub/IA_Aulas/B.Engenharia_de_Atributos/datasets/Musica.csv")
data

,amazed-suprised,happy-pleased,relaxing-clam,quiet-still,sad-lonely,angry-aggresive,Mean_Acc1298_Mean_Mem40_Centroid,Mean_Acc1298_Mean_Mem40_Rolloff,Mean_Acc1298_Mean_Mem40_Flux,Mean_Acc1298_Mean_Mem40_MFCC_0,...,Std_Acc1298_Std_Mem40_MFCC_10,Std_Acc1298_Std_Mem40_MFCC_11,Std_Acc1298_Std_Mem40_MFCC_12,BH_LowPeakAmp,BH_LowPeakBPM,BH_HighPeakAmp,BH_HighPeakBPM,BHSUM1,BHSUM2,BHSUM3
0,0,1,1,0,0,0,0.132498,0.077848,0.229227,0.602629,...,0.197026,0.196244,0.164323,0.030017,0.253968,0.008473,0.240602,0.136735,0.058442,0.107594
1,1,0,0,0,0,1,0.384281,0.355249,0.167190,0.853089,...,0.093526,0.085649,0.025101,0.182955,0.285714,0.156764,0.270677,0.191377,0.153728,0.197951
2,0,1,0,0,0,1,0.541782,0.356491,0.152246,0.791142,...,0.198082,0.108067,0.140574,0.099303,0.142857,0.000000,0.593985,0.105114,0.025555,0.122965
3,0,0,1,0,0,0,0.174288,0.243935,0.254326,0.438987,...,0.235793,0.220195,0.235834,0.024988,0.222222,0.117169,0.210526,0.057288,0.134575,0.091509
4,0,0,0,1,0,0,0.347436,0.155448,0.100047,0.126026,...,0.715683,0.573359,0.412368,0.016398,0.761905,0.081703,0.721805,0.108737,0.172882,0.189934
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587,0,0,1,1,1,0,0.091469,0.014038,0.012550,0.448891,...,0.382121,0.221122,0.193904,0.054434,0.555556,0.320396,0.526316,0.145807,0.001478,0.327572
588,1,0,0,0,1,1,0.456928,0.251840,0.134330,0.881598,...,0.221636,0.217792,0.352805,0.020484,0.206349,0.002236,0.195489,0.157160,0.029046,0.086884
589,0,0,1,1,1,0,0.134808,0.041087,0.048516,0.405211,...,0.141902,0.261306,0.172973,0.019139,0.571429,0.000000,0.541353,0.102674,0.137468,0.128768
590,0,1,1,0,0,0,0.237972,0.302840,0.282430,0.656205,...,0.164731,0.108270,0.068455,0.095590,0.507937,0.156950,0.481203,0.304784,0.102068,0.359101


In [3]:
#y
Classe = data.iloc[:, :5].values
print(f"Classes:\n{Classe}\n")

#X
previsores = data.iloc[:, 6:].values
print(f"Previsores:\n{previsores}\n")


Classes:
[[0 1 1 0 0]
 [1 0 0 0 0]
 [0 1 0 0 0]
 ...
 [0 0 1 1 1]
 [0 1 1 0 0]
 [0 1 0 0 0]]

Previsores:
[[0.132498 0.077848 0.229227 ... 0.136735 0.058442 0.107594]
 [0.384281 0.355249 0.16719  ... 0.191377 0.153728 0.197951]
 [0.541782 0.356491 0.152246 ... 0.105114 0.025555 0.122965]
 ...
 [0.134808 0.041087 0.048516 ... 0.102674 0.137468 0.128768]
 [0.237972 0.30284  0.28243  ... 0.304784 0.102068 0.359101]
 [0.340115 0.155225 0.108587 ... 0.048647 0.131727 0.121288]]



In [4]:
X_train, X_test, y_train, y_test = train_test_split(previsores, Classe, test_size=0.3, random_state=5)

ss = StandardScaler()
X_train_ss = ss.fit_transform(X_train)
X_test_ss = ss.transform(X_test) 

In [34]:
svm = SVC(class_weight="balanced") # O SVC funciona como classificador base para o ClassifierChain; Ele ele é um Support vector machine, ou seja, ele vai criar um hiperplano para separar as classes
#rfc = RandomForestClassifier(class_weight="balanced", n_estimators=10000) # O DecisionTreeClassifier funciona como classificador base para o ClassifierChain; Ele é um algoritmo de árvore de decisão, ou seja, ele vai criar uma árvore de decisão para separar as classes
cc = ClassifierChain(base_estimator=svm, random_state=5) # o classifier chain vai fazer com que o hiperplano seja criado para cada classe e isso é bom ate mesmo para ver relaçoes entre as classes

cc.fit(X_train_ss, y_train)

c:\projetos\GitHub\IA_Aulas\.venv\Lib\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
c:\projetos\GitHub\IA_Aulas\.venv\Lib\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
c:\projetos\GitHub\IA_Aulas\.venv\Lib\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


,estimator estimator: estimatorThe base estimator from which the classifier chain is built.,None
,"order order: array-like of shape (n_outputs,) or 'random', default=NoneIf `None`, the order will be determined by the order of columns inthe label matrix Y.:: order = [0, 1, 2, ..., Y.shape[1] - 1]The order of the chain can be explicitly set by providing a list ofintegers. For example, for a chain of length 5.:: order = [1, 3, 2, 4, 0]means that the first model in the chain will make predictions forcolumn 1 in the Y matrix, the second model will make predictionsfor column 3, etc.If order is `random` a random ordering will be used.",None
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines whether to use cross validated predictions or truelabels for the results of previous estimators in the chain.Possible inputs for cv are:- None, to use true labels when fitting,- integer, to specify the number of folds in a (Stratified)KFold,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.",None
,"chain_method chain_method: {'predict', 'predict_proba', 'predict_log_proba', 'decision_function'} or list of such str's, default='predict'Prediction method to be used by estimators in the chain forthe 'prediction' features of previous estimators in the chain.- if `str`, name of the method;- if a list of `str`, provides the method names in order of preference. The method used corresponds to the first method in the list that is implemented by `base_estimator`... versionadded:: 1.5",'predict'
,"random_state random_state: int, RandomState instance or None, optional (default=None)If ``order='random'``, determines random number generation for thechain order.In addition, it controls the random seed given at each `base_estimator`at each chaining iteration. Thus, it is only used when `base_estimator`exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",5
,"verbose verbose: bool, default=FalseIf True, chain progress is output as each model is completed... versionadded:: 1.2",False
,"base_estimator base_estimator: estimator, default=""deprecated""Use `estimator` instead... deprecated:: 1.7 `base_estimator` is deprecated and will be removed in 1.9. Use `estimator` instead.",SVC(class_weight='balanced')
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


In [35]:
y_pred = cc.predict(X_test_ss)
loss = hamming_loss(y_test, y_pred)
print(loss) # o loss vai representar a média de rótulos incorretos por amostra. Quanto menor o valor, melhor o desempenho do modelo.

0.1797752808988764


In [36]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.72      0.75        60
           1       0.49      0.57      0.53        49
           2       0.71      0.89      0.79        76
           3       0.78      0.84      0.81        45
           4       0.72      0.73      0.73        49

   micro avg       0.69      0.76      0.73       279
   macro avg       0.70      0.75      0.72       279
weighted avg       0.70      0.76      0.73       279
 samples avg       0.60      0.68      0.61       279



c:\projetos\GitHub\IA_Aulas\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\projetos\GitHub\IA_Aulas\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\projetos\GitHub\IA_Aulas\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
